In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

Enabled rmm statistics


In [2]:
%load_ext cudf.pandas

/opt/conda/envs/rapids-25.02/lib/python3.10/site-packages/cudf/pandas/__init__.py:65: UserWarning: cudf.pandas detected an already configured memory resource, ignoring 'CUDF_PANDAS_RMM_MODE'=managed_pool
  warnings.warn(


In [ ]:
%LoadCheckpoint /home/dias-benchmarks/tpch/notebooks/q09/q09_rewrite/checkpoints/pre_cell_6.pickle

In [4]:
%%RecordEvent
%%time
### cell 6 ###

# Narrow down columns to reduce GPU data movement
li = lineitem[[
    'L_PARTKEY', 'L_SUPPKEY', 'L_ORDERKEY',
    'L_EXTENDEDPRICE', 'L_DISCOUNT', 'L_QUANTITY'
]]
fpart = part[['P_PARTKEY']][part.P_NAME.str.contains('ghost')]
sp    = supplier[['S_SUPPKEY', 'S_NATIONKEY']]
na    = nation[['N_NATIONKEY', 'N_NAME']]
ps    = partsupp[['PS_PARTKEY', 'PS_SUPPKEY', 'PS_SUPPLYCOST']]
ord   = orders[['O_ORDERKEY', 'O_ORDERDATE']]

# Chain merges (filter ghost parts → filter partsupp → attach supplier/nation → attach orders),
# compute TMP and O_YEAR, and keep only needed columns
df = (
    li
    .merge(fpart, left_on='L_PARTKEY', right_on='P_PARTKEY')
    .merge(ps,    left_on=['L_PARTKEY', 'L_SUPPKEY'], right_on=['PS_PARTKEY', 'PS_SUPPKEY'])
    .merge(sp,    left_on='L_SUPPKEY',        right_on='S_SUPPKEY')
    .merge(na,    left_on='S_NATIONKEY',      right_on='N_NATIONKEY')
    .merge(ord,   left_on='L_ORDERKEY',       right_on='O_ORDERKEY')
    .assign(
        TMP=lambda x: x.L_EXTENDEDPRICE * (1 - x.L_DISCOUNT) - x.PS_SUPPLYCOST * x.L_QUANTITY,
        O_YEAR=lambda x: x.O_ORDERDATE.dt.year
    )
    [['N_NAME', 'O_YEAR', 'TMP']]  # drop unneeded columns
)

# Cast N_NAME to category for faster grouping
df['N_NAME'] = df['N_NAME'].astype('category')

# Group by nation & year, sum TMP, then sort
total = (
    df.groupby(['N_NAME', 'O_YEAR'], as_index=False)
      .sum()
      .sort_values(['N_NAME', 'O_YEAR'], ascending=[True, False])
)

CPU times: user 86.2 ms, sys: 77 ms, total: 163 ms
Wall time: 199 ms


In [ ]:
%Checkpoint /home/dias-benchmarks/tpch/notebooks/q09/rewritten/o4_mini_high_small_q09/checkpoints/post_cell_6_try_1.pickle